# Retail Customer Intelligence: Segmentation & Sales Forecasting

This notebook is a guided portfolio walkthrough. Run the CLI pipeline first (see project README), then explore its generated outputs.

## 1. Setup
Install dependencies from `requirements.txt` and run from the `data-science-project` folder.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path('data/processed')
REPORTS = Path('reports')
REPORTS.mkdir(parents=True, exist_ok=True)


## 2. Data quality and transaction overview
This section summarizes cleaned transactions created by `src/run_analysis.py`.

In [ ]:
tx = pd.read_csv(DATA / 'clean_transactions.csv', parse_dates=['invoicedate'])
tx.head()


In [ ]:
summary = {
    'rows': len(tx),
    'unique_invoices': tx['invoice'].nunique(),
    'unique_customers_with_id': tx['customer_id'].nunique(),
    'date_min': tx['invoicedate'].min(),
    'date_max': tx['invoicedate'].max(),
    'total_revenue': tx['revenue'].sum(),
}
pd.Series(summary, name='value')


In [ ]:
country_revenue = tx.groupby('country', dropna=False)['revenue'].sum().sort_values(ascending=False).head(10)
country_revenue.sort_values().plot(kind='barh', figsize=(9,5), title='Top 10 countries by revenue')
plt.xlabel('Revenue (dataset currency: GBP)'); plt.tight_layout(); plt.savefig(REPORTS / 'top_countries_revenue.png', dpi=160); plt.show()


## 3. RFM segmentation
Review cluster profiles before assigning business labels. Cluster numbers are arbitrary and do not imply rank.

In [ ]:
rfm = pd.read_csv(DATA / 'customer_rfm_clusters.csv')
profiles = pd.read_csv(DATA / 'cluster_profiles.csv')
profiles


In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
scatter = ax.scatter(rfm['recency_days'], rfm['monetary'], c=rfm['cluster'], alpha=.55, s=18)
ax.set_yscale('log'); ax.set_xlabel('Recency (days; lower is more recent)'); ax.set_ylabel('Monetary value (log scale)')
ax.set_title('Customer clusters: recency vs monetary value'); plt.tight_layout()
plt.savefig(REPORTS / 'rfm_clusters_recency_monetary.png', dpi=160); plt.show()


In [ ]:
selection = pd.read_csv(DATA / 'cluster_selection_metrics.csv')
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(selection['k'], selection['silhouette'], marker='o')
ax.set_xlabel('Number of clusters (k)'); ax.set_ylabel('Silhouette score'); ax.set_title('K-Means candidate evaluation')
plt.tight_layout(); plt.savefig(REPORTS / 'cluster_silhouette.png', dpi=160); plt.show()


## 4. Monthly sales forecasting
The final months are held out chronologically. Do not use random splitting for time-series evaluation.

In [ ]:
monthly = pd.read_csv(DATA / 'monthly_revenue.csv', parse_dates=['invoicedate']).set_index('invoicedate')['revenue']
forecast = pd.read_csv(DATA / 'forecast_holdout.csv', parse_dates=['month'])
metrics = pd.read_csv(DATA / 'forecast_metrics.csv')
metrics


In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(monthly.index, monthly.values, label='Historical revenue', color='gray')
ax.plot(forecast['month'], forecast['actual_revenue'], marker='o', label='Holdout actual')
ax.plot(forecast['month'], forecast['seasonal_naive_forecast'], marker='x', label='Seasonal naive')
if 'sarima_forecast' in forecast:
    ax.plot(forecast['month'], forecast['sarima_forecast'], marker='s', label='SARIMA')
ax.set_title('Monthly revenue: historical series and holdout forecasts'); ax.set_ylabel('Revenue (GBP)'); ax.legend()
plt.tight_layout(); plt.savefig(REPORTS / 'monthly_revenue_forecast.png', dpi=160); plt.show()


## 5. Business interpretation
Use the actual profiles and forecast metrics to answer:
- Which cluster has highest median monetary value?
- Which cluster has the largest recency (possible lapse risk)?
- Which forecast performs better on MAE/RMSE? Is improvement material versus the seasonal baseline?
- What campaign or operational action would you recommend, and what would you measure next?

**Important:** Do not invent cluster labels, metric values, or business impact. Interpret the outputs generated from your own run.